# Netra-One: train and export on Colab T4

This notebook labels the included cleaned image collection, validates YOLO labels, trains YOLOv8n for 35 epochs, exports PyTorch and ONNX weights, creates three visual inference examples, and downloads a submission ZIP.

Before starting, choose **Runtime → Change runtime type → T4 GPU**. Run `make_colab_bundle.py --include-images` on your computer, then upload the resulting `netraone_colab_input.zip` in the next cell. The current local collection is about 106 MiB.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import shutil

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
assert zip_name, 'Upload netraone_colab_input.zip'
project_root = Path('/content/netraone-project')
if project_root.exists():
    shutil.rmtree(project_root)
project_root.mkdir(parents=True)
with ZipFile(zip_name) as bundle:
    bundle.extractall('/content')
project_root = Path('/content/netraone')
assert (project_root / 'train_emergency_yolo.py').exists(), 'Invalid source ZIP'
print('Project ready at', project_root)

In [ ]:
%cd /content/netraone
!pip -q install -r requirements.txt
import torch
assert torch.cuda.is_available(), 'Enable a T4 GPU in Runtime settings.'
print(torch.cuda.get_device_name(0))

## Create and inspect provisional labels

Grounding DINO bootstraps bounding boxes. Review the overlay examples below before training; correct or remove wrong labels in `dataset/labels/` if needed. These auto-labels can contain errors.

In [ ]:
import subprocess

subprocess.run(['python', 'pipeline/auto_label.py', '--device', 'cuda', '--min-score', '0.35'], cwd=project_root, check=True)
subprocess.run(['python', 'pipeline/validate_dataset.py'], cwd=project_root, check=True)

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

review_dir = project_root / 'dataset' / 'review'
previews = []
for class_name in ('indian_ambulance', 'indian_police_vehicle', 'indian_fire_tender'):
    matches = sorted(review_dir.glob(f'{class_name}_*.jpg'))
    if matches:
        previews.append((class_name, Image.open(matches[0]).convert('RGB')))
thumbs = []
for name, image in previews:
    image.thumbnail((420, 280))
    card = Image.new('RGB', (440, 320), 'white')
    card.paste(image, ((440-image.width)//2, 30))
    ImageDraw.Draw(card).text((12, 8), name, fill='black')
    thumbs.append(card)
if thumbs:
    sheet = Image.new('RGB', (440*len(thumbs), 320), '#dddddd')
    for index, thumb in enumerate(thumbs): sheet.paste(thumb, (440*index, 0))
    display(sheet)
else:
    print('No confident boxes; inspect dataset/review and adjust --min-score before training.')

## Train, evaluate, and export

This runs the requested 35 epochs. Validation metrics and the confusion matrix are saved with the model outputs.

In [ ]:
subprocess.run(['python', 'train_emergency_yolo.py', '--device', '0', '--epochs', '35', '--batch', '-1', '--imgsz', '640'], cwd=project_root, check=True)
import json
metrics = json.loads((project_root / 'weights' / 'metrics.json').read_text())
metrics

## Create inference samples and download the submission bundle

The three images below are real model outputs on one collected image from each class. For a traffic footage demo, upload an MP4 and run the CLI shown in `README.md`.

In [ ]:
import json, zipfile
test_dir = project_root / 'test_samples'
test_dir.mkdir(exist_ok=True)
for class_name in ('indian_ambulance', 'indian_police_vehicle', 'indian_fire_tender'):
    source = next((project_root / 'dataset' / 'clean' / class_name).glob('*.jpg'))
    target = test_dir / f'{class_name}_inference.jpg'
    subprocess.run(['python', 'detect_emergency.py', '--source', str(source), '--weights', 'weights/best.pt', '--conf', '0.25', '--output', str(target)], cwd=project_root, check=True)
run_video_demo = False  # Set True and run this cell again to upload a traffic clip
if run_video_demo:
    video_upload = files.upload()
    video_name = next((name for name in video_upload if name.lower().endswith(('.mp4', '.mov', '.avi'))), None)
    assert video_name, 'Upload a short MP4, MOV, or AVI traffic clip'
    video_output = test_dir / 'traffic_sample_detected.mp4'
    subprocess.run(['python', 'detect_emergency.py', '--source', f'/content/{video_name}', '--weights', 'weights/best.pt', '--conf', '0.45', '--output', str(video_output)], cwd=project_root, check=True)
include = ['.gitignore', 'README.md', 'data.yaml', 'requirements.txt', 'train_emergency_yolo.py', 'detect_emergency.py', 'preview_server.py', 'colab_train.ipynb', 'make_colab_bundle.py', 'pipeline', 'dataset/clean/manifest.csv', 'weights/best.pt', 'weights/best.onnx', 'weights/metrics.json', 'test_samples', 'runs/emergency/yolov8n/confusion_matrix.png', 'runs/emergency/yolov8n/results.png']
submission_zip = Path('/content/netraone_submission.zip')
with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for item in include:
        path = project_root / item
        if path.is_file(): archive.write(path, Path('netraone') / path.relative_to(project_root))
        elif path.is_dir():
            for child in path.rglob('*'):
                if child.is_file() and '__pycache__' not in child.parts and child.suffix != '.pyc': archive.write(child, Path('netraone') / child.relative_to(project_root))
print('Submission ZIP:', submission_zip, 'size MiB:', round(submission_zip.stat().st_size/1024**2, 1))
files.download(str(submission_zip))